# Training a diffusion model on Oxford Flowers

The training step in five operations, the generation loop that re-noises on purpose, and 8,189 photographs turned into flowers that do not exist.

**Runs on:** GPU required — about 90 minutes on a Colab T4 &nbsp;·&nbsp; **Slides:** [Chapter 17 — Image Generation](../../../course-web-slides/ch17/index.html) &nbsp;·&nbsp; **Section:** 02 — Diffusion models

---

## The dataset

In [ ]:
import os
import keras

fpath = keras.utils.get_file(
    origin="https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz",
    extract=True)

batch_size, image_size = 32, 128
images_dir = os.path.join(fpath, "jpg")

dataset = keras.utils.image_dataset_from_directory(
    images_dir,
    labels=None,
    image_size=(image_size, image_size),
    crop_to_aspect_ratio=True,
)
dataset = dataset.rebatch(batch_size, drop_remainder=True)

import matplotlib.pyplot as plt
for batch in dataset.take(1):
    fig, axes = plt.subplots(2, 6, figsize=(13, 4.4))
    for ax, im in zip(axes.ravel(), batch):
        ax.imshow(im.numpy().astype("uint8")); ax.axis("off")
    plt.tight_layout(); plt.show()
    break

> ⚠️ **`crop_to_aspect_ratio=True`.** Resizing without it distorts every image, and **distortion in the training set becomes distortion in everything generated** — very hard to diagnose after the fact.

## The model class

In [ ]:
from keras import ops
import numpy as np

class DiffusionModel(keras.Model):
    def __init__(self, image_size, widths, block_depth, **kwargs):
        super().__init__(**kwargs)
        self.image_size = image_size
        self.denoising_model = get_model(image_size, widths, block_depth)
        self.seed_generator = keras.random.SeedGenerator()
        self.loss = keras.losses.MeanAbsoluteError()
        self.normalizer = keras.layers.Normalization()

    def denoise(self, noisy_images, noise_rates, signal_rates):
        pred_noise_masks = self.denoising_model([noisy_images, noise_rates])
        pred_images = (noisy_images - noise_rates * pred_noise_masks) / signal_rates
        return pred_images, pred_noise_masks

**Mean absolute error, not mean squared.** The noise mask is normally distributed; squared error would let a few extreme pixels dominate the gradient.

The `denoise` arithmetic is the exact inverse of the mixing formula: `noisy = signal * image + noise * mask`, so `image = (noisy − noise * mask) / signal`.

## The training step

In [ ]:
class DiffusionModel(DiffusionModel):     # extend the class above
    def call(self, images):
        images = self.normalizer(images)
        noise_masks = keras.random.normal(
            (batch_size, self.image_size, self.image_size, 3),
            seed=self.seed_generator)
        diffusion_times = keras.random.uniform(
            (batch_size, 1, 1, 1), minval=0.0, maxval=1.0,
            seed=self.seed_generator)
        noise_rates, signal_rates = diffusion_schedule(diffusion_times)
        noisy_images = signal_rates * images + noise_rates * noise_masks
        pred_images, pred_noise_masks = self.denoise(
            noisy_images, noise_rates, signal_rates)
        return pred_images, pred_noise_masks, noise_masks

    def compute_loss(self, x, y, y_pred, sample_weight=None, training=True):
        _, pred_noise_masks, noise_masks = y_pred
        return self.loss(noise_masks, pred_noise_masks)

Five operations: normalize, sample **random** diffusion times, compute the rates, add noise, denoise. The loss is one comparison — **all the difficulty is in the forward pass.**

Random times matter: the model will be called at every point of the schedule during generation, so it must be trained across the full spectrum.

## Generation

In [ ]:
class DiffusionModel(DiffusionModel):
    def generate(self, num_images, diffusion_steps):
        noisy_images = keras.random.normal(
            (num_images, self.image_size, self.image_size, 3),
            seed=self.seed_generator)
        step_size = 1.0 / diffusion_steps
        for step in range(diffusion_steps):
            diffusion_times = ops.ones((num_images, 1, 1, 1)) - step * step_size
            noise_rates, signal_rates = diffusion_schedule(diffusion_times)
            pred_images, pred_noises = self.denoise(
                noisy_images, noise_rates, signal_rates)
            next_times = diffusion_times - step_size
            next_noise_rates, next_signal_rates = diffusion_schedule(next_times)
            noisy_images = (next_signal_rates * pred_images
                            + next_noise_rates * pred_noises)
        images = (self.normalizer.mean
                  + pred_images * self.normalizer.variance ** 0.5)
        return ops.clip(images, 0.0, 255.0)

The surprising part: the model predicts the **whole** clean image at every step, and we then **deliberately add back** the noise appropriate to the next time index. Each iteration undoes slightly more than the last.

`diffusion_steps` is a **generation-time** parameter — the same weights sample in 5 steps or 50, trading quality for speed.

## A callback, because there is no metric

In [ ]:
class VisualizationCallback(keras.callbacks.Callback):
    def __init__(self, diffusion_steps=20, num_rows=3, num_cols=6):
        self.diffusion_steps = diffusion_steps
        self.num_rows, self.num_cols = num_rows, num_cols

    def on_epoch_end(self, epoch=None, logs=None):
        generated = self.model.generate(
            num_images=self.num_rows * self.num_cols,
            diffusion_steps=self.diffusion_steps)
        fig, axes = plt.subplots(self.num_rows, self.num_cols,
                                 figsize=(self.num_cols * 2, self.num_rows * 2))
        for ax, im in zip(axes.ravel(), generated):
            ax.imshow(ops.convert_to_numpy(im).astype("uint8")); ax.axis("off")
        plt.suptitle(f"epoch {epoch}"); plt.tight_layout(); plt.show()
        plt.close()

We have no proper metric for image quality, so the practical answer is to look. Chapter 7's callback API, doing something it was not obviously designed for.

## Training

In [ ]:
model = DiffusionModel(image_size, widths=[32, 64, 96, 128],
                       block_depth=2)
model.normalizer.adapt(dataset)          # DO NOT FORGET THIS

model.compile(
    optimizer=keras.optimizers.AdamW(
        learning_rate=keras.optimizers.schedules.InverseTimeDecay(
            initial_learning_rate=1e-3, decay_steps=1000, decay_rate=0.1),
        use_ema=True,
        ema_overwrite_frequency=100,
    ),
)

model.fit(
    dataset,
    epochs=100,
    callbacks=[
        VisualizationCallback(),
        keras.callbacks.ModelCheckpoint("diffusion_model.weights.h5",
                                        save_weights_only=True,
                                        save_best_only=True),
    ],
    verbose=2,
)

> ⚠️ **`model.normalizer.adapt(dataset)`.** Forget it and the noise and the images live on different scales; nothing works, and there is no error message.

On Colab you may hit *"Buffered data was truncated"* because the logs contain images — chain five `fit(..., epochs=20)` calls in five cells instead.

## Two optimizer settings, and what they do

**Learning rate decay** — `InverseTimeDecay` reduces the rate through training.

**Exponential moving average** (Polyak averaging) — keep a running average of the weights and overwrite with it every 100 batches. Helps when the loss landscape is noisy, which a generative objective's is.

Neither changes what the model can represent. Both change **which minimum it settles into**, and for a generative model that is the difference between plausible flowers and coloured smears.

## Sampling at different step counts

In [ ]:
model.load_weights("diffusion_model.weights.h5")

fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
for ax, steps in zip(axes, [3, 5, 10, 20, 50]):
    im = model.generate(num_images=1, diffusion_steps=steps)[0]
    ax.imshow(ops.convert_to_numpy(im).astype("uint8"))
    ax.set_title(f"{steps} steps"); ax.axis("off")
plt.suptitle("Same weights; only the number of denoising steps changes", y=1.04)
plt.tight_layout(); plt.show()

Three steps gives a blurred impression; fifty gives a sharp image. **The quality-versus-latency dial in a production image generator is exactly this parameter.**

---

## What to take away

- Five operations per training step; the loss is one comparison, and diffusion times must be sampled randomly.
- Generation predicts the clean image and then **re-noises** to the next schedule point.
- `normalizer.adapt()` is not optional and fails silently.
- `diffusion_steps` is the quality-versus-latency dial, chosen at generation time.